# 🚀 Fine-Tuning a Sentence-Transformer Model for JD & Resume Matching
### TalentAI Studio • Production ML Training Guide

Welcome! In this Google Colab notebook, you will learn how to fine-tune a state-of-the-art **Sentence-Transformers Bi-Encoder** (`all-MiniLM-L6-v2`) on Job Description (JD) and Resume pairs to predict match scores and rank candidates.

---

### 🧠 Why Bi-Encoders for JD-Resume Matching?
- **Bi-Encoder Architecture:** Encodes the Job Description and the Resume into dense 384-dimensional vector embeddings independently: $\vec{u} = f(\text{JD})$ and $\vec{v} = f(\text{Resume})$.
- **Lightning-Fast Retrieval:** Matching is computed via Cosine Similarity $\cos(\vec{u}, \vec{v}) = \frac{\vec{u} \cdot \vec{v}}{\|\vec{u}\| \|\vec{v}\|}$. This enables sub-millisecond search across thousands of resumes and seamless integration with vector databases like **pgvector** in Supabase!
- **Cost Effective:** Can easily run on a free Google Colab T4 GPU and fine-tune in ~3 minutes.

---
### ⚙️ Step 0: Ensure GPU is Enabled in Colab
1. Click **Runtime** in the top menu.
2. Select **Change runtime type**.
3. Under *Hardware accelerator*, select **T4 GPU** (or any available GPU).
4. Click **Save**.

In [ ]:
# Step 1: Install required libraries
!pip install -q sentence-transformers datasets torch scikit-learn pandas matplotlib seaborn

In [ ]:
# Verify GPU availability
import torch

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ No GPU detected. Running on CPU (training will be slower). Please switch to T4 GPU under Runtime > Change runtime type.")

## 📂 Step 2: Load or Generate the JD-Resume Dataset
You can either upload `jd_resume_dataset.csv` from your TalentAI Studio project, or run the cell below which auto-downloads or creates the mock dataset so this notebook is completely self-contained!

In [ ]:
import pandas as pd
import os

# GitHub raw URL for your cleaned & split Kaggle dataset
GITHUB_DATA_URL = 'https://raw.githubusercontent.com/shyam192837465-byte/ML_Project/shyam/ml_training/dataset/full_cleaned_paired_dataset.csv'

# Check possible local file paths
candidates_paths = [
    'train_dataset.csv',
    'dataset/train_dataset.csv',
    'full_cleaned_paired_dataset.csv',
    'dataset/full_cleaned_paired_dataset.csv',
    'ml_training/dataset/full_cleaned_paired_dataset.csv'
]

dataset_file = None
for p in candidates_paths:
    if os.path.exists(p):
        dataset_file = p
        break

if dataset_file:
    print(f"✅ Loading cleaned Kaggle dataset from local path: {dataset_file}")
    df = pd.read_csv(dataset_file)
else:
    print(f"🌐 Loading cleaned Kaggle dataset directly from your GitHub repo...")
    try:
        df = pd.read_csv(GITHUB_DATA_URL)
        print(f"✅ Successfully loaded from GitHub!")
    except Exception as e:
        print(f"Fallback to synthetic generator ({e})")
        synthetic_data = [
        {
            "id": "PAIR-0001",
            "job_title": "Machine Learning Engineer",
            "job_description": "We need a Machine Learning Engineer to train transformer models (BERT, GPT) using PyTorch, build vector search systems with FAISS/pgvector, and deploy low-latency FastAPI inference microservices with Docker.",
            "candidate_id": "Aarav Sharma",
            "resume_text": "Machine Learning Engineer with 4 years experience. Fine-tuned BERT models in PyTorch for semantic search. Built vector pipelines with pgvector and deployed FastAPI Docker containers handling 5M requests/day.",
            "match_score": 0.94,
            "label": "Strong Match"
        },
        {
            "id": "PAIR-0002",
            "job_title": "Machine Learning Engineer",
            "job_description": "We need a Machine Learning Engineer to train transformer models (BERT, GPT) using PyTorch, build vector search systems with FAISS/pgvector, and deploy low-latency FastAPI inference microservices with Docker.",
            "candidate_id": "Priya Nair",
            "resume_text": "Data Scientist with 3 years experience building predictive models using Python, TensorFlow, and Scikit-learn. Familiar with FastAPI and Docker. Eager to specialize in NLP and Transformers.",
            "match_score": 0.76,
            "label": "Good Match"
        },
        {
            "id": "PAIR-0003",
            "job_title": "Machine Learning Engineer",
            "job_description": "We need a Machine Learning Engineer to train transformer models (BERT, GPT) using PyTorch, build vector search systems with FAISS/pgvector, and deploy low-latency FastAPI inference microservices with Docker.",
            "candidate_id": "Marcus Vance",
            "resume_text": "Python backend developer with 3 years building Django REST APIs and managing PostgreSQL databases. Basic experience with Scikit-learn regression models.",
            "match_score": 0.52,
            "label": "Moderate Match"
        },
        {
            "id": "PAIR-0004",
            "job_title": "Machine Learning Engineer",
            "job_description": "We need a Machine Learning Engineer to train transformer models (BERT, GPT) using PyTorch, build vector search systems with FAISS/pgvector, and deploy low-latency FastAPI inference microservices with Docker.",
            "candidate_id": "Emily Watson",
            "resume_text": "Digital marketer and WordPress content manager with 4 years optimizing landing page conversion, managing client blogs, and running Google Ad campaigns.",
            "match_score": 0.20,
            "label": "Low Match"
        },
        {
            "id": "PAIR-0005",
            "job_title": "Senior Full-Stack Engineer",
            "job_description": "Senior Full-Stack Engineer to architect web apps with React, TypeScript, Tailwind CSS, Node.js, Express, and PostgreSQL. Experience with Docker containerization and CI/CD required.",
            "candidate_id": "Vikram Patel",
            "resume_text": "Full-Stack Engineer with 5 years experience in React, TypeScript, Tailwind CSS, Node.js, and PostgreSQL. Containerized apps with Docker and configured GitHub Actions CI/CD.",
            "match_score": 0.95,
            "label": "Strong Match"
        },
        {
            "id": "PAIR-0006",
            "job_title": "Senior Full-Stack Engineer",
            "job_description": "Senior Full-Stack Engineer to architect web apps with React, TypeScript, Tailwind CSS, Node.js, Express, and PostgreSQL. Experience with Docker containerization and CI/CD required.",
            "candidate_id": "Chloe Bennett",
            "resume_text": "MERN stack developer with 3 years experience in MongoDB, Express, React, and Node.js. Proficient with CSS3 and modern UI design. Basic TypeScript knowledge.",
            "match_score": 0.74,
            "label": "Good Match"
        },
        {
            "id": "PAIR-0007",
            "job_title": "Cloud DevOps Engineer",
            "job_description": "Cloud DevOps Engineer to manage Kubernetes (EKS) clusters, author Terraform infrastructure as code, build GitHub Actions CI/CD pipelines, and configure Prometheus/Grafana monitoring.",
            "candidate_id": "Rahul Verma",
            "resume_text": "Senior DevOps Engineer (CKA certified) with 5 years experience managing multi-region Kubernetes clusters on AWS using Terraform. Automated CI/CD pipelines with GitHub Actions.",
            "match_score": 0.96,
            "label": "Strong Match"
        },
        {
            "id": "PAIR-0008",
            "job_title": "Cloud DevOps Engineer",
            "job_description": "Cloud DevOps Engineer to manage Kubernetes (EKS) clusters, author Terraform infrastructure as code, build GitHub Actions CI/CD pipelines, and configure Prometheus/Grafana monitoring.",
            "candidate_id": "Lucas Meyer",
            "resume_text": "Linux systems administrator with 4 years experience managing Azure VMs, writing Bash automation scripts, and containerizing internal tools with Docker.",
            "match_score": 0.71,
            "label": "Good Match"
        },
        {
            "id": "PAIR-0009",
            "job_title": "Cloud DevOps Engineer",
            "job_description": "Cloud DevOps Engineer to manage Kubernetes (EKS) clusters, author Terraform infrastructure as code, build GitHub Actions CI/CD pipelines, and configure Prometheus/Grafana monitoring.",
            "candidate_id": "Jessica Taylor",
            "resume_text": "UI/UX designer with 3 years experience creating wireframes in Figma and coding landing pages with HTML5, CSS3, and JavaScript.",
            "match_score": 0.22,
            "label": "Low Match"
        },
        {
            "id": "PAIR-0010",
            "job_title": "Senior Data Engineer",
            "job_description": "Senior Data Engineer to build batch and streaming data pipelines with Python, Apache Spark, and SQL. Manage Airflow DAGs and architect data models in Snowflake.",
            "candidate_id": "Ananya Gupta",
            "resume_text": "Data Engineer with 4.5 years experience building PySpark data pipelines processing 3TB/day into Snowflake. Orchestrated 60+ Airflow DAGs with custom operators.",
            "match_score": 0.94,
            "label": "Strong Match"
        },
        {
            "id": "PAIR-0011",
            "job_title": "Senior Data Engineer",
            "job_description": "Senior Data Engineer to build batch and streaming data pipelines with Python, Apache Spark, and SQL. Manage Airflow DAGs and architect data models in Snowflake.",
            "candidate_id": "Rohan Deshmukh",
            "resume_text": "BI Analyst with 4 years writing advanced SQL queries, building Tableau executive dashboards, and doing basic data wrangling in Python with Pandas.",
            "match_score": 0.67,
            "label": "Moderate Match"
        },
        {
            "id": "PAIR-0012",
            "job_title": "Senior Backend Engineer (Go)",
            "job_description": "Backend Engineer to build high-throughput microservices in Go (Golang), implement gRPC interfaces, Kafka event streams, and optimize PostgreSQL databases.",
            "candidate_id": "Dmitri Volkov",
            "resume_text": "Distributed systems engineer with 5 years building high-performance Go microservices. Specialized in event-driven streaming with Kafka, gRPC, and PostgreSQL.",
            "match_score": 0.96,
            "label": "Strong Match"
        },
        {
            "id": "PAIR-0013",
            "job_title": "Senior Backend Engineer (Go)",
            "job_description": "Backend Engineer to build high-throughput microservices in Go (Golang), implement gRPC interfaces, Kafka event streams, and optimize PostgreSQL databases.",
            "candidate_id": "Amina Bello",
            "resume_text": "Backend developer with 4 years building enterprise microservices in Java, Spring Boot, and Kafka. Working on Go personal projects.",
            "match_score": 0.72,
            "label": "Good Match"
        }
        ]
        df = pd.DataFrame(synthetic_data)

print(f"✅ Successfully loaded dataset with {len(df)} samples!")
df[['job_title', 'candidate_id', 'match_score', 'label']].head()

### 📊 Inspecting the Score Distribution
Let's see the distribution of match scores across roles.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.hist(df['match_score'], bins=8, color='#6366f1', edgecolor='black', alpha=0.8)
plt.title('Distribution of JD-Resume Match Scores', fontsize=13, fontweight='bold')
plt.xlabel('Match Score (0.0 to 1.0)')
plt.ylabel('Count')
plt.grid(axis='y', alpha=0.3)
plt.show()

## 🧬 Step 3: Preparing Input Data for Sentence-Transformers
Sentence-Transformers requires `InputExample` objects where:
- `texts = [job_description, resume_text]`
- `label = float(match_score)` (continuous similarity from 0.0 to 1.0)

In [ ]:
from sentence_transformers import InputExample
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

# Train / Validation Split (80% train, 20% validation)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

train_examples = [
    InputExample(texts=[row['job_description'], row['resume_text']], label=float(row['match_score']))
    for _, row in train_df.iterrows()
]

val_examples = [
    InputExample(texts=[row['job_description'], row['resume_text']], label=float(row['match_score']))
    for _, row in val_df.iterrows()
]

# Create PyTorch DataLoader
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=4)

print(f"Train samples: {len(train_examples)}")
print(f"Validation samples: {len(val_examples)}")

## 🤖 Step 4: Initializing the Bi-Encoder Model & Loss
We use `sentence-transformers/all-MiniLM-L6-v2`:
- Highly optimized 6-layer MiniLM model.
- 384-dimensional dense vectors.
- Fast inference (over 1000 sentences/sec on T4 GPU).
- **Loss Function**: `losses.CosineSimilarityLoss(model)` minimizes Mean Squared Error between the cosine similarity of the embeddings and the target score.

In [ ]:
from sentence_transformers import SentenceTransformer, losses, evaluation

MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
print(f"Loading base model: {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME)

# Loss Function: CosineSimilarityLoss for continuous score regression (0.0 to 1.0)
train_loss = losses.CosineSimilarityLoss(model)

# Evaluator for monitoring validation performance during training
evaluator = evaluation.EmbeddingSimilarityEvaluator.from_input_examples(
    val_examples,
    name='jd-resume-eval',
    show_progress_bar=False
)

print("✅ Model and Evaluator initialized successfully!")

## 🏋️ Step 5: Fine-Tuning the Model on Colab GPU
Now we run the training loop using `model.fit()`. On a Colab T4 GPU, 4 epochs take **less than 60 seconds**!

In [ ]:
import math

num_epochs = 4
warmup_steps = math.ceil(len(train_dataloader) * num_epochs * 0.1)  # 10% warmup
output_dir = './talentai_jd_matcher_model'

print(f"Starting fine-tuning for {num_epochs} epochs...")

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=num_epochs,
    evaluation_steps=20,
    warmup_steps=warmup_steps,
    output_path=output_dir,
    show_progress_bar=True
)

print(f"\n🎉 Training Complete! Fine-tuned model saved to: {output_dir}")

## 🧪 Step 6: Testing & Evaluating Predictions
Let's load our newly fine-tuned model and test it on validation samples to inspect predicted vs actual match scores.

In [ ]:
from sentence_transformers import util

# Load best saved checkpoint
best_model = SentenceTransformer(output_dir)

print("=" * 85)
print(f"{'Candidate':<20} | {'Expected':<10} | {'Model Pred':<12} | {'Difference':<10} | Status")
print("=" * 85)

for _, row in val_df.iterrows():
    # Encode JD and Resume into vectors
    jd_emb = best_model.encode(row['job_description'], convert_to_tensor=True)
    res_emb = best_model.encode(row['resume_text'], convert_to_tensor=True)
    
    # Compute Cosine Similarity
    pred_score = util.cos_sim(jd_emb, res_emb).item()
    expected = float(row['match_score'])
    diff = abs(pred_score - expected)
    status = "✅ Close" if diff < 0.20 else "⚠️ Gap"
    
    print(f"{row['candidate_id']:<20} | {expected:<10.2f} | {pred_score:<12.2f} | {diff:<10.2f} | {status}")

## 🏆 Step 7: Live Demonstration - Matching & Ranking Resumes for a New JD
Now let's simulate the real recruiter experience! We provide a brand-new Job Description and rank 4 candidates from 1st to 4th place.

In [ ]:
def rank_candidates(job_description, candidates_list, model):
    """
    Encodes the job description once, encodes all candidates, and ranks them by cosine similarity.
    """
    jd_embedding = model.encode(job_description, convert_to_tensor=True)
    
    ranked_results = []
    for cand in candidates_list:
        resume_emb = model.encode(cand['resume'], convert_to_tensor=True)
        similarity = util.cos_sim(jd_embedding, resume_emb).item()
        score_percent = round(similarity * 100, 1)
        
        ranked_results.append({
            'name': cand['name'],
            'role': cand.get('role', 'Candidate'),
            'similarity_score': similarity,
            'match_percent': f"{score_percent}%"
        })
    
    # Sort descending by similarity score
    ranked_results.sort(key=lambda x: x['similarity_score'], reverse=True)
    return ranked_results

# Target Job Description
new_jd = """
Looking for a Senior Machine Learning Engineer with 4+ years experience fine-tuning LLMs and transformers using PyTorch. Must have production experience building vector search with pgvector/Pinecone and deploying low-latency FastAPI endpoints on Docker.
"""

# Sample candidate pool
test_candidates = [
    {
        'name': 'Aarav Sharma',
        'role': 'Senior ML Engineer',
        'resume': 'Machine Learning Engineer with 4 years experience. Fine-tuned BERT and LLMs in PyTorch. Implemented vector search with pgvector and deployed Dockerized FastAPI services.'
    },
    {
        'name': 'Marcus Vance',
        'role': 'Backend Developer',
        'resume': 'Backend Python developer with 3 years building Django REST APIs, PostgreSQL queries, and basic Scikit-learn data processing.'
    },
    {
        'name': 'Priya Nair',
        'role': 'Data Scientist',
        'resume': 'Data Scientist with 3.5 years experience in predictive modeling with Python, TensorFlow, and Scikit-learn. Familiar with FastAPI and Docker.'
    },
    {
        'name': 'Emily Watson',
        'role': 'Digital Marketer',
        'resume': 'Digital marketing specialist with 4 years experience optimizing WordPress landing pages, SEO campaigns, and Google Analytics.'
    }
]

results = rank_candidates(new_jd, test_candidates, best_model)

print("\n🎯 Candidate Rankings for the Machine Learning Role:")
print("-" * 60)
medals = ['🥇 #1', '🥈 #2', '🥉 #3', '  #4']
for i, res in enumerate(results):
    print(f"{medals[i]} {res['name']} ({res['role']}) -> Match Score: {res['match_percent']}")

## 🚀 Step 8: Deploying as a FastAPI Microservice
Here is how to serve the trained model with FastAPI so your TalentAI React app can call it via HTTP POST!

In [ ]:
# Example FastAPI server snippet (save as server.py in your backend)
fastapi_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer, util
from typing import List

app = FastAPI(title="TalentAI Matching Service")
model = SentenceTransformer("./talentai_jd_matcher_model")

class CandidateItem(BaseModel):
    id: str
    name: str
    resume_text: str

class MatchRequest(BaseModel):
    job_description: str
    candidates: List[CandidateItem]

@app.post("/api/match-and-rank")
def match_and_rank(req: MatchRequest):
    jd_emb = model.encode(req.job_description, convert_to_tensor=True)
    ranked = []
    for c in req.candidates:
        res_emb = model.encode(c.resume_text, convert_to_tensor=True)
        sim = util.cos_sim(jd_emb, res_emb).item()
        score = round(sim * 100, 1)
        ranked.append({"id": c.id, "name": c.name, "score": score})
    
    ranked.sort(key=lambda x: x["score"], reverse=True)
    return {"ranked_candidates": ranked}
'''

with open('serve_model.py', 'w') as f:
    f.write(fastapi_code)

print("✅ Saved serve_model.py microservice script!")

### 📦 Step 9: Download Your Fine-Tuned Model
Run this cell to zip your trained model so you can download it to your local machine or upload to Google Drive / Hugging Face Hub.

In [ ]:
!zip -r talentai_jd_matcher_model.zip ./talentai_jd_matcher_model

try:
    from google.colab import files
    files.download('talentai_jd_matcher_model.zip')
    print("✅ Download started!")
except Exception as e:
    print("Model zipped. You can download talentai_jd_matcher_model.zip from the Colab file explorer on the left.")